# Unit II — AI in Drug Safety & Pharmacovigilance
**Pharmakaksha · Learn · Grow · Achieve** — BP703T AI in Clinical Applications (B.Pharm Sem VII, PCI NEP 2020)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ajreddys/learn-pharmakaksha-site/blob/main/content/BP703T%20AI%20in%20Clinical%20Applications%20%28Theory%29/BP703T_Unit2_AI_in_Pharmacovigilance.ipynb)

Every example from the Unit II notes, slides and video, in order, then lab exercises with solutions. Data: `faers_demo.csv`, `faers_drug.csv`, `faers_reac.csv`, `anticoag_cohort.csv` (fictional teaching data).

## Using learn.pharmakaksha.com
- **First run is slow.** Python starts inside your browser; the first cell can take about 10 seconds. Later cells are fast.
- **Kernel indicator.** The circle at the top right is filled while Python is busy. Wait for it to clear.
- **Stuck?** Use *Kernel → Restart Kernel*, then run the cells again from the top.
- **Saving.** Your changes are saved in this browser only. Use *File → Download* to keep a copy.
- **Start over.** Delete your copy from the file list on the left and reopen the notebook from the course folder.

In [ ]:
import os
import numpy as np
import pandas as pd
BASE = "https://raw.githubusercontent.com/ajreddys/learn-pharmakaksha-site/main/content/BP703T%20AI%20in%20Clinical%20Applications%20%28Theory%29/"

def load(name):
    """Read a practice CSV: local copy first, then the course's GitHub folder."""
    if os.path.exists(name):
        return pd.read_csv(name)
    try:
        return pd.read_csv(BASE + name)
    except Exception:
        from pyodide.http import open_url   # inside JupyterLite
        return pd.read_csv(open_url(BASE + name))

## 2.1–2.2 FAERS-style data: three linked tables
FAERS ships quarterly files linked by `primaryid`: DEMO (who and when), DRUG (which drugs, with role PS = primary suspect, SS = secondary suspect, C = concomitant) and REAC (MedDRA preferred terms). Ours are fictional, with six made-up drug names.

In [ ]:
demo = load("faers_demo.csv")
drug = load("faers_drug.csv")
reac = load("faers_reac.csv")
print(len(demo), "reports |", len(drug), "drug rows |", len(reac), "reaction rows")
demo.head()

In [ ]:
drug["role_cod"].value_counts()

## 2.3 Frequency analysis
Join the primary-suspect drug to each reaction, then count.

In [ ]:
ps = drug[drug["role_cod"] == "PS"][["primaryid", "drugname"]]
pairs = ps.merge(reac, on="primaryid").drop_duplicates()
print(ps["drugname"].value_counts(), "\n")
print(reac["pt"].value_counts())

In [ ]:
pd.crosstab(pairs["drugname"], pairs["pt"])

## 2.3 Signal detection: the 2×2 table, PRR and ROR
| | Event | Other events |
|---|---|---|
| Drug | a | b |
| All other drugs | c | d |

PRR = [a/(a+b)] / [c/(c+d)];  ROR = (a·d)/(b·c). **Evans criteria:** PRR ≥ 2, χ² ≥ 4, a ≥ 3.

In [ ]:
reports = pairs.groupby("primaryid").agg(drug=("drugname", "first"), pts=("pt", set))

def disproportionality(drug_name, event, reports=reports):
    is_d = reports["drug"] == drug_name
    is_e = reports["pts"].apply(lambda s: event in s)
    a = int((is_d & is_e).sum()); b = int((is_d & ~is_e).sum())
    c = int((~is_d & is_e).sum()); d = int((~is_d & ~is_e).sum())
    prr = (a / (a + b)) / (c / (c + d))
    se = np.sqrt(1/a - 1/(a + b) + 1/c - 1/(c + d))
    ror = (a * d) / (b * c)
    se_r = np.sqrt(1/a + 1/b + 1/c + 1/d)
    n = a + b + c + d
    expected = [(a+b)*(a+c)/n, (a+b)*(b+d)/n, (c+d)*(a+c)/n, (c+d)*(b+d)/n]
    chi2 = sum((o - e)**2 / e for o, e in zip([a, b, c, d], expected))
    return {"drug": drug_name, "event": event, "a": a, "b": b, "c": c, "d": d,
            "PRR": round(prr, 2), "PRR_lo": round(np.exp(np.log(prr) - 1.96*se), 2),
            "PRR_hi": round(np.exp(np.log(prr) + 1.96*se), 2),
            "ROR": round(ror, 2), "ROR_lo": round(np.exp(np.log(ror) - 1.96*se_r), 2),
            "ROR_hi": round(np.exp(np.log(ror) + 1.96*se_r), 2), "chi2": round(chi2, 1)}

pd.DataFrame([disproportionality("Zentarin", "Pancreatitis"),
              disproportionality("Ardoxin", "Nausea")])

In [ ]:
screen = pd.DataFrame([disproportionality(d, e)
                       for d in ps["drugname"].unique() for e in reac["pt"].unique()])
signals = screen[(screen["a"] >= 3) & (screen["PRR"] >= 2) & (screen["chi2"] >= 4)]
signals[["drug", "event", "a", "PRR", "PRR_lo", "PRR_hi", "chi2"]]

A signal is a **hypothesis**, not proof. Next step: case-by-case causality assessment (e.g. WHO-UMC), literature, and checking for bias.

## 2.7 Bias in spontaneous reports: the notoriety effect

In [ ]:
import matplotlib.pyplot as plt
z = pairs[(pairs["drugname"] == "Zentarin") & (pairs["pt"] == "Pancreatitis")].merge(demo, on="primaryid")
z["month"] = z["event_dt"].str[5:7].astype(int)
counts = z["month"].value_counts().sort_index()
print(counts.tolist())
plt.bar(counts.index, counts.values, color=["#E07A2E" if m in (9, 10) else "#1F8A70" for m in counts.index])
plt.axvline(8.5, ls="--", color="#12303A"); plt.xlabel("Month of 2025"); plt.ylabel("Reports")
plt.title("Zentarin–pancreatitis reports by month"); plt.show()

In [ ]:
pd.crosstab(z["month"].isin([9, 10]), z["reporter_type"], normalize="index").round(2)

## 2.4 Logistic regression for ADR risk
1,200 hospital patients on an anticoagulant (fictional). Outcome: major bleed within 90 days.

In [ ]:
c = load("anticoag_cohort.csv")
print("Major bleed rate:", round(c["major_bleed"].mean(), 3))
c.head()

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import confusion_matrix, roc_auc_score

features = ["age", "crcl_mL_min", "nsaid", "antiplatelet", "liver_disease", "prior_bleed"]
X, y = c[features], c["major_bleed"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model = make_pipeline(StandardScaler(), LogisticRegression()).fit(X_tr, y_tr)
prob = model.predict_proba(X_te)[:, 1]
print("Test patients:", len(y_te), "| bleeds:", int(y_te.sum()))
print("ROC AUC:", round(roc_auc_score(y_te, prob), 3))

## 2.5–2.6 Confusion matrix, sensitivity, specificity, precision, recall
The default threshold of 0.5 is a poor choice when only about 1 in 10 patients bleed.

In [ ]:
def evaluate(threshold):
    tn, fp, fn, tp = confusion_matrix(y_te, (prob >= threshold).astype(int)).ravel()
    return {"threshold": threshold, "TP": tp, "FP": fp, "FN": fn, "TN": tn,
            "accuracy": round((tp + tn) / len(y_te), 3),
            "sensitivity (recall)": round(tp / (tp + fn), 3),
            "specificity": round(tn / (tn + fp), 3),
            "precision": round(tp / (tp + fp), 3) if tp + fp else None}

pd.DataFrame([evaluate(t) for t in [0.5, 0.2, 0.1]])

## 2.7 Confounding by indication
In observational data, doctors gave Drug A to older patients with weaker kidneys. Does Drug A cause bleeding?

In [ ]:
print(c.groupby("drug")[["age", "crcl_mL_min", "prior_bleed", "major_bleed"]].mean().round(2))
tab = pd.crosstab(c["drug"], c["major_bleed"])
crude_or = (tab.loc["A", 1] * tab.loc["B", 0]) / (tab.loc["A", 0] * tab.loc["B", 1])
print("\nCrude OR (A vs B):", round(crude_or, 2))

In [ ]:
c["drugA"] = (c["drug"] == "A").astype(int)
adj = LogisticRegression(C=1e6, max_iter=5000).fit(c[features + ["drugA"]], c["major_bleed"])
pd.Series(np.exp(adj.coef_[0]), index=features + ["drugA"]).round(2).rename("adjusted OR")

Adjusting for age, kidney function and the other risk factors shrinks Drug A's odds ratio from about 3.7 to about 1.25. Most of the crude difference came from **who** received Drug A.

## Worked example: a pharmacist's bleeding-risk review

In [ ]:
new = pd.DataFrame([[78, 35, 1, 0, 0, 1], [78, 35, 0, 0, 0, 1], [50, 95, 0, 0, 0, 0]], columns=features,
                   index=["Mr. S as admitted", "Mr. S, NSAID stopped", "Ms. P"])
new["predicted_risk"] = model.predict_proba(new[features])[:, 1].round(3)
new

---
# Lab exercises
1. Compute PRR and ROR (with 95% CI) for Belmira–Hepatotoxicity. Does it meet the Evans criteria?
2. Remove the reports from months 9–10 for Zentarin–Pancreatitis and recompute the PRR. What does the notoriety effect do to the signal?
3. Find the highest threshold that gives sensitivity ≥ 0.9 for major bleeding. Report specificity and precision.
4. Which adjusted odds ratio is the largest among the clinical risk factors? Explain it in one clinical sentence.

## Solutions

In [ ]:
# Lab 1
disproportionality("Belmira", "Hepatotoxicity")

In [ ]:
# Lab 2: drop Zentarin–Pancreatitis reports from months 9–10
month = demo.set_index("primaryid")["event_dt"].str[5:7].astype(int)
spike_ids = z.loc[z["month"].isin([9, 10]), "primaryid"]
reports_clean = reports.drop(index=spike_ids)
print("PRR with spike:   ", disproportionality("Zentarin", "Pancreatitis")["PRR"])
print("PRR without 9–10: ", disproportionality("Zentarin", "Pancreatitis", reports_clean)["PRR"])

In [ ]:
# Lab 3
for t in np.arange(0.30, 0.0, -0.005):
    r = evaluate(round(t, 3))
    if r["sensitivity (recall)"] >= 0.9:
        print(r); break

In [ ]:
# Lab 4
odds = pd.Series(np.exp(adj.coef_[0]), index=features + ["drugA"]).drop("drugA")
print(odds.round(2).sort_values(ascending=False))

**Teaching data only.** Drug names, reports and patients are fictional.